<a href="https://colab.research.google.com/github/shivanivanitha0-dev/mini-project2/blob/main/fdc(1)).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q streamlit pyngrok tensorflow scikit-learn pandas numpy matplotlib seaborn

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving flight_delay_classification_app_bundle.zip to flight_delay_classification_app_bundle (3).zip


In [ ]:
!pip install -q streamlit pyngrok tensorflow scikit-learn pandas numpy matplotlib seaborn

import os

if os.path.exists("/content/flight_delay_app.py"):
    os.rename("/content/flight_delay_app.py", "/content/app.py")

print("Application file exists:", os.path.exists("/content/app.py"))

Application file exists: True


In [ ]:
from pyngrok import ngrok
import getpass
import subprocess
import time
import os

# Retrieve token from environment/variables if possible, otherwise prompt the user
try:
    # Try using the previously set token if available in local notebook variables
    if 'token' in globals() and token:
        ngrok.set_auth_token(token)
        print("Using previously configured ngrok token.")
    else:
        user_token = getpass.getpass("Enter your ngrok token: ")
        ngrok.set_auth_token(user_token)
except Exception as e:
    print(f"Could not configure token automatically: {e}")

# Start Streamlit
log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# Wait for the server to start
time.sleep(10)

# Create the public browser link
try:
    # Disconnect previous connections if any
    ngrok.kill()
    tunnel = ngrok.connect(8501)
    print("YOUR APPLICATION LINK:")
    print(tunnel.public_url)
except Exception as e:
    print("\nFailed to connect ngrok tunnel.")
    print("Please verify your authtoken at https://dashboard.ngrok.com/get-started/your-authtoken")
    print(f"Error details: {e}")

Enter your ngrok token: ··········
Could not configure token automatically: An error occurred when saving the auth token: b'Authtoken already set in configuration file: /root/.config/ngrok/ngrok.yml\n'
YOUR APPLICATION LINK:
https://altitude-glacier-ion.ngrok-free.dev


In [ ]:
import os

print("app.py exists:", os.path.exists("/content/app.py"))
print("Files in /content:", os.listdir("/content")[:30])

app.py exists: True
Files in /content: ['.config', 'india_flight_delays_2025.csv', 'streamlit.log', 'flight_delay_classification_app_bundle (2).zip', 'airline_delay.csv', 'flight_delay_classification_app_bundle.zip', 'requirements.txt', 'flight_delay_classification_app_bundle (3).zip', 'README.txt', 'flight_delay_classification_app_bundle (1).zip', 'india_flight_delay_model_input (1).csv', 'app.py', 'sample_data']


In [ ]:
!pip install -q streamlit pyngrok

In [ ]:
import subprocess
import time
import requests

# Start Streamlit
process = subprocess.Popen(
    [
        "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=open("/content/streamlit.log", "w"),
    stderr=subprocess.STDOUT
)

# Check whether Streamlit starts
for i in range(30):
    try:
        response = requests.get(
            "http://127.0.0.1:8501/_stcore/health",
            timeout=2
        )
        if response.status_code == 200:
            print("SUCCESS: Streamlit is running!")
            break
    except requests.RequestException:
        time.sleep(2)
else:
    print("Streamlit did not start. Error log:")
    print(open("/content/streamlit.log").read()[-5000:])

SUCCESS: Streamlit is running!


In [ ]:
import os
import zipfile
from pyngrok import ngrok

# 1. Unzip the uploaded bundle if app.py is missing
zip_path = "/content/flight_delay_classification_app_bundle.zip"
if not os.path.exists("/content/app.py") and os.path.exists(zip_path):
    print("Extracting application bundle...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("/content/")
    print("Extraction complete.")

# 2. Forcefully kill any existing ngrok processes in the background
print("Stopping existing ngrok sessions...")
ngrok.kill()

# 3. Create the new tunnel
try:
    tunnel = ngrok.connect(8501, "http")
    print("\nOPEN YOUR FLIGHT DELAY APPLICATION HERE:")
    print(tunnel.public_url)
except Exception as e:
    print(f"\nError starting tunnel: {e}")

Stopping existing ngrok sessions...

OPEN YOUR FLIGHT DELAY APPLICATION HERE:
https://altitude-glacier-ion.ngrok-free.dev


In [ ]:
import os
import time
import subprocess
import requests

APP_PATH = "/content/app.py"
LOG_PATH = "/content/streamlit.log"

# 1. Check application file
if not os.path.exists(APP_PATH):
    print("ERROR: app.py is missing!")
    print("Files currently in /content:")
    print(os.listdir("/content")[:50])
else:
    print("SUCCESS: app.py exists")

    # 2. Check whether Streamlit is already running
    try:
        r = requests.get(
            "http://127.0.0.1:8501/_stcore/health",
            timeout=2
        )
        if r.status_code == 200:
            print("SUCCESS: Streamlit is already running!")
        else:
            raise Exception("Streamlit is not healthy")

    except Exception:
        # 3. Start Streamlit and capture errors
        log = open(LOG_PATH, "w")
        process = subprocess.Popen(
            [
                "python", "-m", "streamlit", "run", APP_PATH,
                "--server.port", "8501",
                "--server.address", "0.0.0.0",
                "--server.headless", "true"
            ],
            stdout=log,
            stderr=subprocess.STDOUT
        )

        # 4. Wait up to 60 seconds
        for attempt in range(30):
            time.sleep(2)

            try:
                r = requests.get(
                    "http://127.0.0.1:8501/_stcore/health",
                    timeout=2
                )
                if r.status_code == 200:
                    print("SUCCESS: Streamlit is running!")
                    break
            except requests.RequestException:
                pass

            if process.poll() is not None:
                print("ERROR: Streamlit stopped unexpectedly.")
                break

        # 5. Display the real error if startup failed
        try:
            r = requests.get(
                "http://127.0.0.1:8501/_stcore/health",
                timeout=2
            )
            running = r.status_code == 200
        except requests.RequestException:
            running = False

        if not running:
            print("\n--- STREAMLIT ERROR LOG ---")
            log.flush()
            with open(LOG_PATH, "r") as f:
                print(f.read()[-8000:])
        else:
            print("Now you can create the ngrok link.")

SUCCESS: app.py exists
SUCCESS: Streamlit is already running!


In [ ]:
from pyngrok import ngrok
import requests

# Confirm the application is responding
response = requests.get("http://127.0.0.1:8501/_stcore/health")
print("Streamlit status:", response.text)

# Create a public link
tunnel = ngrok.connect(8501, "http")

print("https://altitude-glacier-ion.ngrok-free.dev")
print(tunnel.public_url)

Streamlit status: ok
https://altitude-glacier-ion.ngrok-free.dev
https://altitude-glacier-ion.ngrok-free.dev


In [ ]:
import os
from google.colab import files

target_file = "894233989_32020_056_airline_delay_causes.csv.zip"
found_path = None

# Search recursively for the file inside /content
for root, dirs, files_list in os.walk("/content"):
    if target_file in files_list:
        found_path = os.path.join(root, target_file)
        break

if found_path:
    print(f"Found file at: {found_path}. Starting download...")
    files.download(found_path)
else:
    # Fallback: search for any CSV or zip file that might be the dataset
    print(f"Could not find '{target_file}' directly.")
    print("Here are the files currently available in /content:")
    for root, dirs, files_list in os.walk("/content"):
        for f in files_list:
            if "airline" in f.lower() or "delay" in f.lower():
                print(f"Potential match: {os.path.join(root, f)}")

Could not find '894233989_32020_056_airline_delay_causes.csv.zip' directly.
Here are the files currently available in /content:
Potential match: /content/india_flight_delays_2025.csv
Potential match: /content/flight_delay_classification_app_bundle (2).zip
Potential match: /content/airline_delay.csv
Potential match: /content/flight_delay_classification_app_bundle.zip
Potential match: /content/flight_delay_classification_app_bundle (3).zip
Potential match: /content/flight_delay_classification_app_bundle (1).zip
Potential match: /content/india_flight_delay_model_input (1).csv


In [ ]:
import pandas as pd
import urllib.request
from google.colab import files

url = "https://www.openintro.org/data/index.php/csv/csv/csv/csv/csv/csv/csv/rda/csv/rda/tab-delimited/rda/rda/rda/R/rda/rda/csv/rda/rda/csv/csv/airline_delay.csv"
output_path = ""

try:
    # Use urllib with a standard web browser User-Agent header to avoid HTTP 406 errors
    req = urllib.request.Request(
        url,
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    )
    with urllib.request.urlopen(req) as response:
        with open(output_path, 'wb') as out_file:
            out_file.write(response.read())

    # Load the downloaded CSV file into pandas
    df = pd.read_csv(output_path)

    print("Dataset downloaded successfully!")
    print("Dataset shape:", df.shape)
    print("Columns:", df.columns.tolist())

    # Download to your local system
    files.download(output_path)

except Exception as e:
    print(f"An error occurred: {e}")

An error occurred: [Errno 2] No such file or directory: ''
